# 02 · Campaign — single-mutation scoring + ProteinMPNN CDR redesign → candidate CSV

**Standard slot:** *design campaign.* **For Project 15 this means:** the maturation campaign —
(a) **score single CDR mutations** with ESM-1v and AbLang (framework fixed), (b) run **ProteinMPNN CDR
redesigns** (framework fixed) to catch multi-residue loop changes, and (c) assemble a **candidate
mutation set** as a results CSV (D2).

**Compute reality (be honest):** ESM-1v and AbLang scoring is **light** — CPU/T4 is fine, and this is
the cheap, high-value part. ProteinMPNN CDR redesign is also cheap. The **heavy** step (notebook 03/04)
is AF2-Multimer pose checking — keep N small and **batch** it. Colab **T4–Pro** is the realistic tier.
This notebook runs on the **mock** backend so the plumbing executes anywhere; all numbers are
RANKING scores, flagged SYNTHETIC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything.

In [ ]:
import requests

# Pinned upstreams for affinity maturation (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "ESM / ESM-1v (protein LM mutation scoring)": "https://github.com/facebookresearch/esm",
    "AbLang (antibody LM)": "https://github.com/oxpig/AbLang",
    "ProteinMPNN (CDR redesign, framework fixed)": "https://github.com/dauparas/ProteinMPNN",
    "ColabFold (AF2-Multimer pose check)": "https://github.com/sokrypton/ColabFold",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: AbLang2 supersedes AbLang — VERIFY the current public repo and pin it (MANUAL.md §2).")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters

Re-state the parent + the campaign scale. The single-mutation scan is exhaustive over the CDR positions
(cheap). ProteinMPNN redesigns add multi-residue loop variants. Keep the eventual *tested* set SMALL —
diversity in scoring, then aggressive filtering, then a short list to the bench.

In [ ]:
from maturation_tools import (example_parent_sequence, cdr_positions, EXAMPLE_FRAMEWORK)

ANTIGEN = "ANTIGEN"
parent = example_parent_sequence()       # EXAMPLE placeholder — use your verified chain on a real run
FRAMEWORK = EXAMPLE_FRAMEWORK

# Single-mutation scan is exhaustive over CDR positions; MPNN adds N redesigns per CDR.
MPNN_PER_CDR = 8         # small mock N; real ProteinMPNN run can sample more (cheap)
TOOL = "mock"            # -> "esm1v"/"ablang"/"proteinmpnn" on Colab (light; CPU/T4 fine)

print("parent length:", len(parent), "aa")
print("CDR spans:", cdr_positions(FRAMEWORK))
print(f"campaign: exhaustive single-mutation scan + {MPNN_PER_CDR} MPNN redesigns/CDR  (tool={TOOL})")
print("Diversity BEFORE filtering: score broadly here, filter hard in nb 03, test a SMALL set.")

## (a) Single-mutation scoring (ESM-1v + AbLang)

`score_single_mutations()` tries the 19 non-WT residues at every CDR position (framework fixed) and
scores each with the ESM-1v Δ-log-likelihood proxy + AbLang naturalness. Switch `TOOL` to `"esm1v"` /
`"ablang"` on Colab to run for real (the functions raise a clear, actionable `NotImplementedError` with
the TODO until then). **These are RANKING scores, not affinities.**

In [ ]:
import pandas as pd
from maturation_tools import score_single_mutations

singles = score_single_mutations(parent, framework=FRAMEWORK, tool=TOOL, antigen=ANTIGEN)
rows = [v.as_row() for v in singles]
single_df = pd.DataFrame(rows)
print(f"scored {len(single_df)} single CDR mutations (ESM-1v + AbLang, {('SYNTHETIC' if TOOL=='mock' else 'real')})")
# Favored single mutations (esm1v > 0) are the maturation candidates; show the strongest few.
fav = single_df.sort_values("esm1v", ascending=False).head(8)
print("\ntop single mutations by ESM-1v (RANK only, NOT KD):")
fav[["design_id", "cdr", "mutations", "esm1v", "ablang"]]

## (b) ProteinMPNN CDR redesigns (framework FIXED)

`mpnn_cdr_redesign()` proposes new sequences for ONE CDR loop at a time while holding **all** framework
positions (and the other CDRs) fixed — exploring multi-residue changes that single-mutation scanning
misses. On a real run this uses a ProteinMPNN design mask so only the chosen CDR varies. Survivors must
still pass the pose check + developability scan downstream.

In [ ]:
from maturation_tools import mpnn_cdr_redesign

redesigns = []
for cdr in ("CDR1", "CDR2", "CDR3"):
    redesigns += mpnn_cdr_redesign(parent, cdr=cdr, n=MPNN_PER_CDR, framework=FRAMEWORK,
                                   tool=TOOL, antigen=ANTIGEN)
print(f"ProteinMPNN redesigns (framework FIXED): {len(redesigns)} "
      f"({MPNN_PER_CDR} per CDR x 3 CDRs)")
redesign_df = pd.DataFrame([v.as_row() for v in redesigns])
redesign_df[["design_id", "cdr", "mutations", "ablang"]].head()

## (c) Assemble the candidate mutation set → CSV

Combine the single mutations and the CDR redesigns into one candidate pool and write the campaign CSV.
We keep the columns the filter + analysis need. This pool is deliberately broad at SCORING time; the
aggressive filtering in notebook 03 and the SMALL final list happen later — diversity before
filtering.

In [ ]:
camp = pd.concat([single_df, redesign_df], ignore_index=True)
cols = ["design_id", "source", "antigen", "cdr", "mutations",
        "esm1v", "ablang", "pae_interaction", "scrmsd", "n_liabilities", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?", bool(camp["synthetic"].fillna(False).all()) if "synthetic" in camp else "n/a",
      "(mock => all scores are EXAMPLE_DATA ranking values, not affinities)")
camp.head()

## Quick campaign sanity look

Before filtering, eyeball the distributions: the ESM-1v score (how many mutations the LM favors at all)
and the AbLang naturalness. On the **mock** backend these are SYNTHETIC and only show the plumbing; on a
real run they tell you whether there is signal worth carrying into the pose check.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(camp["esm1v"].dropna(), bins=20); ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].set_title("ESM-1v Δ-LL (mock)"); ax[0].set_xlabel(">0 = favored over WT")
ax[1].hist(camp["ablang"].dropna(), bins=20); ax[1].set_title("AbLang naturalness (mock)")
ax[1].set_xlabel("0-1")
plt.suptitle("Campaign scores — SYNTHETIC (mock); for plumbing only; RANKS, not affinities")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock distributions are SYNTHETIC ranking scores — real shape comes from ESM-1v/AbLang.")

## D2 checklist
- [ ] `results/campaign.csv`: the candidate pool (single mutations + CDR redesigns), one row per
      candidate, with ESM-1v / AbLang ranking scores.
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md` (ESM, AbLang/AbLang2, MPNN).
- [ ] Design log: parent, framework, CDR spans, scan settings, MPNN params, seed, tool/version.
- [ ] (Real run) ESM-1v ensemble + AbLang scored; ProteinMPNN CDR redesigns with framework masked.
- [ ] Reminder recorded: these are **ranking** scores; the SMALL tested set + controls come later.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter (`design_type="antibody"`).